# Naive Bayes: Probabilistic Classification Made SimpleNaive Bayes is a family of simple yet powerful probabilistic classifiers based on applying Bayes' theorem with strong (naive) independence assumptions between the features. Despite their simplicity, they often perform remarkably well in many real-world scenarios, particularly in text classification and high-dimensional settings.> "All models are wrong, but some are useful." — George Box. The naive assumption is often wrong, but Naive Bayes remains surprisingly useful.## What You'll Learn- **Bayes Theorem** refresher with practical examples- The **'naive' assumption** of conditional independence — and why it's called naive- **5 variants** of Naive Bayes: GaussianNB, MultinomialNB, BernoulliNB, ComplementNB, CategoricalNB- When to use each variant- **Implementation** with scikit-learn- **Text classification** with MultinomialNB & ComplementNB- **Decision boundary** visualization for GaussianNB- **Probability calibration** — how well do NB probabilities reflect reality?- **Real-world wine classification** with confusion matrix- **Comparison** against Logistic Regression and Decision Trees

---## 1. Bayes Theorem RefresherBayes Theorem describes the probability of an event based on prior knowledge of conditions related to the event:$$P(A | B) = \frac{P(B | A) \cdot P(A)}{P(B)}$$Where:- $P(A | B)$ is the **posterior** probability of class A given evidence B- $P(B | A)$ is the **likelihood** of evidence B given class A- $P(A)$ is the **prior** probability of class A- $P(B)$ is the **marginal** probability of evidence B### Intuitive Example: Spam DetectionSuppose we know:- 30% of emails are spam → $P(Spam) = 0.3$- 80% of spam contains the word 'free' → $P(free | Spam) = 0.8$- 5% of legitimate emails contain 'free' → $P(free | Ham) = 0.05$If an email contains 'free', what's the probability it's spam?$$P(Spam | free) = \frac{P(free | Spam) \cdot P(Spam)}{P(free)} = \frac{0.8 \times 0.3}{0.8 \times 0.3 + 0.05 \times 0.7} = \frac{0.24}{0.275} \approx 0.873$$There's an 87.3% chance it's spam. This is the posterior probability — our updated belief after seeing the evidence.

In [ ]:
# Bayes Theorem: concrete calculation
P_spam = 0.3        # prior
P_ham = 0.7         # prior
P_free_given_spam = 0.8   # likelihood
P_free_given_ham = 0.05   # likelihood

# Marginal probability of seeing 'free'
P_free = P_free_given_spam * P_spam + P_free_given_ham * P_ham

# Posterior
P_spam_given_free = (P_free_given_spam * P_spam) / P_free

print(f'P(Spam | "free") = {P_spam_given_free:.4f}')
print(f'There is a {P_spam_given_free:.1%} chance the email is spam.')

---## 2. The 'Naive' Assumption: Conditional IndependenceFor classification, we want: $P(y | x_1, x_2, ..., x_n)$ — the probability of class $y$ given all features $x_1$ through $x_n$.Using Bayes Theorem:$$P(y | x_1, ..., x_n) = \frac{P(x_1, ..., x_n | y) \cdot P(y)}{P(x_1, ..., x_n)}$$The challenge: computing $P(x_1, ..., x_n | y)$ requires modeling the joint distribution of all features — which is exponential in the number of features.### The Naive AssumptionWe assume features are **conditionally independent** given the class:$$P(x_1, ..., x_n | y) = P(x_1 | y) \times P(x_2 | y) \times ... \times P(x_n | y)$$This simplifies dramatically: we just need to estimate $P(x_i | y)$ for each feature independently!### Why is it called 'Naive'?Features are almost never conditionally independent in real data. For example, in spam detection, if you see the word 'free', you're more likely to see the word 'click' too. They're not independent.### Why does it still work?1. **Classification doesn't need accurate probabilities** — it just needs the correct ranking of classes. Even if probabilities are wrong, the argmax may still be right.2. **Feature dependencies often cancel out** — over-estimated probabilities in one direction may be balanced by other dependent features.3. **Works great with high dimensions** — where more complex models struggle with the curse of dimensionality.4. **Very little data needed** — each feature's distribution is estimated independently, so you need far fewer samples.

---## 3. Types of Naive BayesDifferent variants handle different types of feature distributions:| Variant | Feature Type | Distribution Assumption | Best For ||---|---|---|---|| **GaussianNB** | Continuous | Gaussian (Normal) | Real-valued features, sensor data || **MultinomialNB** | Counts | Multinomial | Text classification, word counts || **BernoulliNB** | Binary | Bernoulli | Presence/absence features || **ComplementNB** | Counts | Multinomial (complement) | Imbalanced text data || **CategoricalNB** | Categorical | Categorical | Discrete categories |Let's explore each one in detail.

---### GaussianNBAssumes each feature follows a Gaussian (normal) distribution per class:$$P(x_i | y) = \frac{1}{\sqrt{2\pi\sigma_{y,i}^2}} \exp\left(-\frac{(x_i - \mu_{y,i})^2}{2\sigma_{y,i}^2}\right)$$For each class $y$ and feature $i$, we estimate:- $\mu_{y,i}$: mean of feature $i$ for class $y$- $\sigma_{y,i}^2$: variance of feature $i$ for class $y$These are stored in `GaussianNB.theta_` (means) and `GaussianNB.var_` (variances) after fitting.

In [ ]:
# Visualize how GaussianNB models probability distributions per class
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import norm
from sklearn.datasets import make_blobs, make_classification, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import (GaussianNB, MultinomialNB, BernoulliNB,
                                  ComplementNB, CategoricalNB)
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             brier_score_loss)
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
import time

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All imports loaded successfully.')

In [ ]:
# Generate 3 well-separated Gaussian clusters
X_dist, y_dist = make_blobs(n_samples=600, centers=3, n_features=2,
                              cluster_std=1.8, random_state=42)

X_train_dist, X_test_dist, y_train_dist, y_test_dist = train_test_split(
    X_dist, y_dist, test_size=0.3, random_state=42
)

gnb_dist = GaussianNB()
gnb_dist.fit(X_train_dist, y_train_dist)

print(f'Training accuracy: {gnb_dist.score(X_train_dist, y_train_dist):.3f}')
print(f'Test accuracy: {gnb_dist.score(X_test_dist, y_test_dist):.3f}')

# The learned parameters: means and variances per class per feature
print(f'\\nLearned means (theta_):\\n{gnb_dist.theta_}')
print(f'\\nLearned variances (var_):\\n{gnb_dist.var_}')

# Visualize the Gaussian fits per class for both features
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for feature_idx, ax in enumerate(axes):
    for cls in range(3):
        cls_data = X_train_dist[y_train_dist == cls, feature_idx]
        ax.hist(cls_data, bins=20, alpha=0.4, density=True,
                label=f'Class {cls} (data)', color=f'C{cls}')
        mu = gnb_dist.theta_[cls, feature_idx]
        sigma = np.sqrt(gnb_dist.var_[cls, feature_idx])
        x_range = np.linspace(cls_data.min() - 1, cls_data.max() + 1, 200)
        ax.plot(x_range, norm.pdf(x_range, mu, sigma),
                linewidth=2.5, color=f'C{cls}',
                label=f'Class {cls} (Gaussian fit, $\\mu$={mu:.2f}, $\\sigma$={sigma:.2f})')
    ax.set_xlabel(f'Feature {feature_idx + 1}')
    ax.set_ylabel('Density')
    ax.set_title(f'Probability Distribution: Feature {feature_idx + 1}')
    ax.legend(fontsize=8)

plt.suptitle('GaussianNB: How Each Class Models Each Feature as a Gaussian', fontsize=14)
plt.tight_layout()
plt.show()

---### MultinomialNBAssumes features represent **counts** (e.g., word frequencies). The probability of a feature vector given a class follows a multinomial distribution:$$P(x | y) \propto \prod_{i=1}^{n} \theta_{y,i}^{x_i}$$Where $\theta_{y,i}$ is the probability of feature $i$ occurring in class $y$, estimated as:$$\theta_{y,i} = \frac{N_{y,i} + \alpha}{N_y + \alpha \cdot n}$$$\alpha$ is the smoothing parameter (default $\alpha=1.0$, Laplace smoothing) to handle zero counts.### ComplementNBA variant of MultinomialNB designed for **imbalanced datasets**. Instead of computing $P(x | y)$, it computes $P(x | \bar{y})$ — the probability of features given the **complement** (all other classes). This naturally focuses on the features that distinguish a class from the rest.### BernoulliNBFor **binary features** (present/absent, true/false). Each feature is modeled as a Bernoulli distribution: $P(x_i | y) = p_{y,i}^{x_i} (1 - p_{y,i})^{1 - x_i}$.For text classification, BernoulliNB uses **binary term occurrence** (whether a word appears or not) rather than term frequency.### CategoricalNBFor **categorical features** where each feature can take one of several discrete values. Each feature $i$ has its own set of probabilities $P(x_i = k | y)$ for each category $k$.**Important**: CategoricalNB expects features encoded as integers (0, 1, 2, ...). Use `OrdinalEncoder` or `KBinsDiscretizer` to convert continuous features.

---## 4. Text Classification with Naive BayesNaive Bayes is a cornerstone of text classification. Let's simulate a document classification task with synthetic word-count data and compare how the different variants perform.

In [ ]:
# Generate synthetic text-like count data
# We create documents where different word groups are indicative of different classes
rng = np.random.RandomState(42)
n_docs = 500
n_words = 100  # vocabulary size

# Class 0: words 0-19 have elevated counts
# Class 1: words 20-39 have elevated counts
X_text = rng.poisson(lam=2, size=(n_docs, n_words))
y_text = np.zeros(n_docs, dtype=int)
y_text[n_docs // 2:] = 1

# Inject signal: class-specific high-frequency words
X_text[:n_docs // 2, :20] += rng.poisson(lam=6, size=(n_docs // 2, 20))
X_text[n_docs // 2:, 20:40] += rng.poisson(lam=6, size=(n_docs // 2, 20))

X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42
)

print(f'Training set: {X_train_txt.shape}')
print(f'Test set: {X_test_txt.shape}')
print(f'\\nAverage word count per document: {X_text.sum(axis=1).mean():.0f}')
print(f'Class distribution: {np.bincount(y_text)}')

In [ ]:
# Compare text-oriented NB variants
text_models = {
    'MultinomialNB': MultinomialNB(),
    'ComplementNB': ComplementNB(),
    'BernoulliNB': BernoulliNB(binarize=0.5)  # binarize at low threshold
}

print('Text Classification Performance:')
print('=' * 50)
for name, model in text_models.items():
    model.fit(X_train_txt, y_train_txt)
    train_acc = model.score(X_train_txt, y_train_txt)
    test_acc = model.score(X_test_txt, y_test_txt)
    print(f'{name:15s}: Train={train_acc:.3f}, Test={test_acc:.3f}')

# For MultinomialNB, examine which 'words' (features) are most indicative of each class
mnb = MultinomialNB()
mnb.fit(X_train_txt, y_train_txt)

# feature_log_prob_ contains log P(word | class) for each class
# Higher values = more indicative of that class
top_n = 10
print(f'\\nTop {top_n} words for Class 0:', np.argsort(mnb.feature_log_prob_[0])[-top_n:])
print(f'Top {top_n} words for Class 1:', np.argsort(mnb.feature_log_prob_[1])[-top_n:])
print('\\nNotice: Class 0 prefers words 0-19, Class 1 prefers words 20-39 as expected.')

---## 5. Comparing All Naive Bayes VariantsLet's systematically compare all 5 NB variants across multiple random datasets. Each variant expects different data types, so we'll transform appropriately.

In [ ]:
# Systematic comparison across multiple random seeds
results = []
n_runs = 5

for seed in range(n_runs):
    X, y = make_classification(n_samples=500, n_features=20, n_informative=15,
                                n_redundant=5, flip_y=0.05, random_state=seed)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed + 100
    )

    # For MultinomialNB & ComplementNB: shift to non-negative counts
    X_train_pos = X_train - X_train.min(axis=0) + 0.1
    X_test_pos = X_test - X_test.min(axis=0) + 0.1

    # For BernoulliNB: binarize at 0
    X_train_bin = (X_train > 0).astype(int)
    X_test_bin = (X_test > 0).astype(int)

    # For CategoricalNB: discretize into 5 bins
    disc = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
    X_train_cat = disc.fit_transform(X_train).astype(int)
    X_test_cat = disc.transform(X_test).astype(int)

    estimators = [
        ('GaussianNB', GaussianNB(), X_train, X_test),
        ('MultinomialNB', MultinomialNB(), X_train_pos, X_test_pos),
        ('ComplementNB', ComplementNB(), X_train_pos, X_test_pos),
        ('BernoulliNB', BernoulliNB(), X_train_bin, X_test_bin),
        ('CategoricalNB', CategoricalNB(), X_train_cat, X_test_cat),
    ]

    for name, model, X_tr, X_te in estimators:
        model.fit(X_tr, y_train)
        acc = accuracy_score(y_test, model.predict(X_te))
        results.append({'Model': name, 'Accuracy': acc, 'Seed': seed})

df_results = pd.DataFrame(results)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_results, x='Model', y='Accuracy', palette='viridis',
            errorbar=('ci', 95))
plt.title('Naive Bayes Variants: Accuracy Comparison (5 random seeds)', fontsize=14)
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('Summary Statistics:')
print(df_results.groupby('Model')['Accuracy'].describe().round(4))

---## 6. Decision Boundary VisualizationLet's visualize how GaussianNB partitions the feature space. The decision boundaries are quadratic because each class has its own covariance matrix (a diagonal Gaussian).

In [ ]:
# Create a 2D dataset for visualization
X_vis, y_vis = make_classification(
    n_samples=400, n_features=2, n_informative=2,
    n_redundant=0, n_clusters_per_class=1,
    flip_y=0.1, class_sep=1.2, random_state=42
)

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_vis, y_vis, test_size=0.3, random_state=42
)

gnb_viz = GaussianNB()
gnb_viz.fit(X_train_v, y_train_v)

# Create mesh grid
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 0.02),
    np.arange(y_min, y_max, 0.02)
)
Z = gnb_viz.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(12, 5))

# Left: decision boundary
plt.subplot(1, 2, 1)
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
scatter_train = plt.scatter(
    X_train_v[:, 0], X_train_v[:, 1], c=y_train_v,
    marker='o', edgecolors='k', s=60, label='Train',
    cmap=plt.cm.RdYlBu, alpha=0.8
)
scatter_test = plt.scatter(
    X_test_v[:, 0], X_test_v[:, 1], c=y_test_v,
    marker='s', edgecolors='k', s=60, label='Test',
    cmap=plt.cm.RdYlBu, alpha=0.8
)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('GaussianNB Decision Boundary')
plt.legend()

# Right: predicted probability magnitudes
plt.subplot(1, 2, 2)
Z_prob = gnb_viz.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z_prob = Z_prob.reshape(xx.shape)
plt.contourf(xx, yy, Z_prob, levels=20, cmap='RdYlBu', alpha=0.7)
plt.colorbar(label='P(Class 1)')
plt.scatter(
    X_test_v[:, 0], X_test_v[:, 1], c=y_test_v,
    marker='s', edgecolors='k', s=80, cmap=plt.cm.RdYlBu
)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('GaussianNB Predicted Probabilities')

plt.tight_layout()
plt.show()

print(f'Test accuracy: {gnb_viz.score(X_test_v, y_test_v):.3f}')
print(f'\\nPredicted probabilities for first 5 test samples:')
print(gnb_viz.predict_proba(X_test_v[:5]))

---## 7. Probability CalibrationNaive Bayes tends to produce probabilities that are **over-confident** (too close to 0 or 1) due to the independence assumption causing repeated multiplication of correlated evidence. Let's compare with Logistic Regression which is naturally well-calibrated.

In [ ]:
# Generate a moderately complex dataset
X_cal, y_cal = make_classification(
    n_samples=1000, n_features=5, n_informative=3,
    flip_y=0.05, random_state=42
)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cal, y_cal, test_size=0.3, random_state=42
)

# Fit models
gnb_cal = GaussianNB()
gnb_cal.fit(X_train_c, y_train_c)
prob_pos_gnb = gnb_cal.predict_proba(X_test_c)[:, 1]

lr_cal = LogisticRegression(max_iter=5000, random_state=42)
lr_cal.fit(X_train_c, y_train_c)
prob_pos_lr = lr_cal.predict_proba(X_test_c)[:, 1]

# Plot calibration curves
fig, ax = plt.subplots(figsize=(8, 6))

CalibrationDisplay.from_predictions(
    y_test_c, prob_pos_gnb, n_bins=10, ax=ax,
    name='GaussianNB', marker='o', linewidth=2
)
CalibrationDisplay.from_predictions(
    y_test_c, prob_pos_lr, n_bins=10, ax=ax,
    name='Logistic Regression', marker='s', linewidth=2
)
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', alpha=0.7)
ax.set_title('Probability Calibration Curves', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

# Brier score (lower is better)
print(f'GaussianNB Brier score:       {brier_score_loss(y_test_c, prob_pos_gnb):.4f}')
print(f'Logistic Regression Brier:    {brier_score_loss(y_test_c, prob_pos_lr):.4f}')
print(f'\\nThe Brier score measures mean squared error between predicted probabilities and actual outcomes.')
print('Lower is better. LR is typically better calibrated than NB.')

---## 8. Real-World Example: Wine Dataset ClassificationThe Wine dataset contains 178 samples of 3 wine types, each described by 13 chemical features (alcohol, malic acid, ash, etc.). Let's apply GaussianNB and evaluate performance.

In [ ]:
# Load Wine dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print(f'Dataset shape: {X_wine.shape}')
print(f'Classes: {target_names}')
print(f'Samples per class: {np.bincount(y_wine)}')
print(f'Feature names: {feature_names}')

# Standardize (GaussianNB can handle unnormalized data, but it can help)
# Note: GaussianNB is scale-invariant in terms of predictions (it models each feature
# independently with its own Gaussian), so scaling doesn't change predictions.
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)

gnb_wine = GaussianNB()
gnb_wine.fit(X_train_w, y_train_w)
y_pred_w = gnb_wine.predict(X_test_w)

print(f'\\nTest Accuracy: {accuracy_score(y_test_w, y_pred_w):.3f}')
print(f'\\nClassification Report:')
print(classification_report(y_test_w, y_pred_w, target_names=target_names))

In [ ]:
# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_estimator(
    gnb_wine, X_test_w, y_test_w,
    display_labels=target_names, cmap='Blues',
    ax=axes[0], normalize=None,
    values_format='d'
)
axes[0].set_title('Confusion Matrix (counts)', fontsize=13)

ConfusionMatrixDisplay.from_estimator(
    gnb_wine, X_test_w, y_test_w,
    display_labels=target_names, cmap='Blues',
    ax=axes[1], normalize='true',
    values_format='.2f'
)
axes[1].set_title('Confusion Matrix (row-normalized)', fontsize=13)

plt.tight_layout()
plt.show()

# Cross-validation
cv_scores = cross_val_score(GaussianNB(), X_wine, y_wine, cv=5)
print(f'5-Fold Cross-Validation Scores: {cv_scores}')
print(f'Mean CV accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})')

---## 9. Comparison: Naive Bayes vs Logistic Regression vs Decision TreeLet's compare GaussianNB against two other popular classifiers across datasets with different characteristics.

In [ ]:
# Define models to compare
models_to_compare = {
    'GaussianNB': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

# Test on 3 different synthetic datasets
datasets = {
    'Linear Separable': make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=0, flip_y=0.02, random_state=42
    ),
    'Noisy (flip_y=0.3)': make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=0, flip_y=0.3, random_state=42
    ),
    'High-Dimensional': make_classification(
        n_samples=300, n_features=50, n_informative=10,
        n_redundant=5, random_state=42
    )
}

comparison_data = []

for dataset_name, (X_ds, y_ds) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_ds, y_ds, test_size=0.3, random_state=42
    )
    for model_name, model in models_to_compare.items():
        model.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, model.predict(X_te))
        comparison_data.append({
            'Dataset': dataset_name,
            'Model': model_name,
            'Accuracy': acc
        })

df_cmp = pd.DataFrame(comparison_data)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_cmp, x='Dataset', y='Accuracy', hue='Model', palette='Set2')
plt.title('Model Comparison Across Different Dataset Types', fontsize=14)
plt.ylabel('Test Accuracy')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Print detailed results
print('Detailed Results:')
print(df_cmp.pivot_table(index='Dataset', columns='Model', values='Accuracy'))

In [ ]:
# Training time comparison
X_big, y_big = make_classification(n_samples=5000, n_features=100, random_state=42)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_big, y_big, test_size=0.3, random_state=42
)

print('Training Time Comparison (5000 samples, 100 features):')
print('=' * 50)

for name, model in models_to_compare.items():
    start = time.time()
    model.fit(X_tr_b, y_tr_b)
    fit_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_te_b)
    pred_time = time.time() - start

    acc = accuracy_score(y_te_b, y_pred)
    print(f'{name:25s}: fit={fit_time:.4f}s, predict={pred_time:.4f}s, accuracy={acc:.3f}')

print(f'\\nNote: GaussianNB is typically the fastest to train (just computing class-wise means and variances).')

---## 10. Pros and Cons of Naive Bayes### Pros- **Extremely fast**: Training is linear in the number of samples and features. Prediction is also linear.- **Works well with high dimensions**: Handles thousands of features easily.- **Works with small data**: Each feature's distribution is estimated independently, so you need fewer samples than more complex models.- **Handles missing elegantly**: When making predictions, features that are missing simply don't contribute to the product.- **Good for categorical features**: CategoricalNB and MultinomialNB naturally handle discrete features.- **Probabilistic predictions**: Provides calibrated (though not perfectly) probability estimates.- **Online learning**: Can be updated incrementally (via `partial_fit`).### Cons- **Independence assumption rarely holds**: Features are almost always correlated. This can lead to overconfident predictions.- **Zero-frequency problem**: If a feature value doesn't appear in training for a class, the entire probability becomes zero. (Solved by smoothing like Laplace/Additive smoothing.)- **Poor with correlated features**: Unlike tree-based methods or regularized models, NB doesn't handle feature interactions well.- **Not ideal for regression**: While there are variants, NB is primarily a classification tool.- **Sensitive to data representation**: For continuous data, GaussianNB assumes normality, which may not hold.

---## 11. Practice ExercisesTest your understanding with these exercises:---**Exercise 1: Bayes Theorem Calculation**A medical test for a disease has 99% sensitivity (true positive rate) and 95% specificity (true negative rate). The disease affects 1% of the population. If a person tests positive, what is the probability they actually have the disease? Calculate this using Bayes Theorem.---**Exercise 2: GaussianNB Parameter Exploration**Modify the `var_smoothing` parameter of GaussianNB (default = 1e-9). Try values [1e-9, 1e-6, 1e-3, 0.1, 1.0] on the Wine dataset and plot how test accuracy changes. What does var_smoothing do and why would you increase it?---**Exercise 3: Comparing Smoothing in MultinomialNB**The `alpha` parameter controls Laplace/Lidstone smoothing in MultinomialNB. Train MultinomialNB with alpha values [0.001, 0.01, 0.1, 1.0, 10.0] on the text classification data and show how accuracy and the top indicative words change.---**Exercise 4: BernoulliNB Threshold**BernoulliNB's `binarize` parameter controls the threshold for converting features to binary. On the Wine dataset (use KBinsDiscretizer to convert to binary features if needed), test binarize thresholds [0.0, 0.5, 1.0, 2.0] and compare performance. Which threshold works best?---**Exercise 5: ComplementNB for Imbalanced Data**Create a highly imbalanced classification dataset (e.g., 95% class 0, 5% class 1) using make_classification with weights=[0.95, 0.05]. Compare MultinomialNB and ComplementNB on this dataset. Which one performs better on the minority class? Use classification_report to examine per-class metrics.